# Gold — physical_feriado_dia

**KPI 10 (Squad 3 — Negócio/Técnica, Luiz Henrique):**
"Enriquecimento: cruzar dia da venda com feriados nacionais para
flag 'venda_em_feriado'" — "Analisar se feriados aumentam ou
diminuem o volume de itens vendidos".

**Por que este notebook é separado de `gold_physical_itens_venda_caixa`:**
granularidade diferente. A tabela de itens tem grão
**produto+loja+mês** (exigência oficial do KPI 6). Feriado é por
natureza **por dia**. Juntar as duas granularidades numa tabela só
faz o valor de feriado se repetir por produto (ou por dia, se
forçado ao grão mais fino), inflando qualquer soma feita no Looker.
Notebook e tabela Gold próprios resolvem isso de forma limpa, e
seguem o mesmo padrão do resto do projeto (1 notebook por tabela
Gold).

**Saída:** tabela `gold_physical_feriado_dia`, grão **loja + dia**.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = get_adls_options()

# Modo de escrita da Gold:
#   "overwrite" -> reprocessamento completo
#   "append"    -> carga incremental (novos dados chegando no raw)
# IMPORTANTE: a Gold no SQL Server usa sempre mode="overwrite" via
# write_sql_table_typed (substitui a tabela inteira a cada execução),
# independente deste modo -- este controla apenas a escrita em Delta.
GOLD_WRITE_MODE = "overwrite"

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, count, countDistinct, round as spark_round,
    year, month, quarter, when, col, lit, to_date, expr, first
)

## Leitura da Silver + resolução de loja/data

Mesma preparação usada em `gold_physical_itens_venda_caixa`
(Silver de itens válidos + `id_loja`/`dt_venda` resolvidos via
Bronze de `vendas_caixa`) — necessária aqui também porque o KPI 10
precisa de receita por dia por loja, que depende dessas mesmas
colunas.

In [0]:
df_itens_silver = (
    read_delta(SILVER_ITENS_VENDA_CAIXA_PATH, adls_options)
    .filter(col("flag_fk_invalido") == False)
    .filter(col("flag_quantidade_invalido") == False)
    .filter(col("flag_preco_invalido") == False)
)

print(f"Itens válidos (Silver): {df_itens_silver.count():,}")

df_vendas_dim = (
    read_delta(BRONZE_VENDAS_CAIXA_PATH, adls_options)
    .select(
        col("id_transacao"),
        expr("TRY_CAST(id_loja AS BIGINT)").alias("id_loja"),
        col("dt_venda"),
    )
    .dropDuplicates(["id_transacao"])
)

df_itens = (
    df_itens_silver
    .drop("id_loja", "dt_venda", "tipo_pagamento")
    .join(df_vendas_dim, on="id_transacao", how="inner")
    .withColumn("dt_venda_date", to_date(col("dt_venda")))
    .withColumn("ano",      year(col("dt_venda_date")))
    .withColumn("mes",      month(col("dt_venda_date")))
    .withColumn("trimestre", quarter(col("dt_venda_date")))
)

print(f"Itens após JOIN com vendas_caixa: {df_itens.count():,}")

### Integridade referencial: `id_loja` órfão

Mesma checagem aplicada em `gold_physical_itens_venda_caixa` —
necessária de novo aqui porque este notebook lê a Silver
independentemente, não reaproveita o `df_itens` do outro notebook.

In [0]:
df_ids_lojas_validas = (
    read_delta(SILVER_LOJAS_PATH, adls_options)
    .select("id_loja")
    .distinct()
)

df_itens = (
    df_itens
    .join(
        df_ids_lojas_validas.withColumn("_loja_existe", lit(True)),
        on="id_loja",
        how="left",
    )
    .withColumn("flag_id_loja_orfao", col("_loja_existe").isNull())
    .drop("_loja_existe")
)

qtd_orfaos = df_itens.filter(col("flag_id_loja_orfao")).count()
total_pre_filtro = df_itens.count()

if qtd_orfaos > 0:
    print(f"[ALERTA] {qtd_orfaos:,} de {total_pre_filtro:,} item(ns) "
          f"com id_loja orfao -- excluidos, ver metrica de DQ.")
else:
    print("[OK] Nenhum id_loja orfao encontrado.")

registrar_metrica_dq(
    tabela="physical_feriado_dia",
    regra="01_id_loja_orfao_referencial",
    qtd_registros_afetados=qtd_orfaos,
    qtd_registros_total=total_pre_filtro,
    adls_options=adls_options,
)

df_itens = df_itens.filter(~col("flag_id_loja_orfao"))

## Verificação de pré-condição

In [0]:
verificar_destino_limpo(
    GOLD_FERIADO_DIA_PATH,
    adls_options,
    permitir_existente=(GOLD_WRITE_MODE == "overwrite"),
)

## KPI 10 — Enriquecimento: flag `venda_em_feriado`

Feriados nacionais brasileiros calculados deterministicamente
(sem API externa, sem passo manual) via `gerar_df_feriados_brasil`
— cobre automaticamente todos os anos presentes nos dados.

In [0]:
anos_presentes = [
    row["ano"] for row in df_itens.select("ano").distinct().collect()
]
df_feriados = gerar_df_feriados_brasil(spark, anos=anos_presentes)

df_itens = (
    df_itens
    .join(
        df_feriados.withColumn("_eh_feriado", lit(True)),
        df_itens["dt_venda_date"] == df_feriados["data_feriado"],
        how="left",
    )
    .withColumn("venda_em_feriado", col("_eh_feriado").isNotNull())
    .drop("data_feriado", "_eh_feriado")
)

qtd_feriado = df_itens.filter(col("venda_em_feriado")).count()
print(f"{qtd_feriado:,} item(ns) vendido(s) em feriado "
      f"(anos: {anos_presentes}).")

## Agregação — granularidade DIA

Grão: **loja + dia**. Como dia é a unidade atômica aqui, o Looker
pode somar com segurança em qualquer nível (mês, trimestre, ano)
sem duplicar nada.

In [0]:
df_kpi_feriado_dia = (
    df_itens
    .groupBy("id_loja", "ano", "mes", "trimestre", "dt_venda_date", "venda_em_feriado")
    .agg(
        spark_round(spark_sum("valor_item_analitico"), 2).alias("receita_dia"),
        countDistinct("id_transacao").alias("qtd_transacoes_dia"),
        first("nome_feriado", ignorenulls=True).alias("nome_feriado"),
    )
)

print(f"KPI 10 (granularidade dia): {df_kpi_feriado_dia.count():,} linha(s) "
      f"— uma por loja+dia com venda.")
display(
    df_kpi_feriado_dia
    .filter(col("venda_em_feriado"))
    .orderBy("id_loja", "dt_venda_date")
    .limit(10)
)

## Escrita no Delta (Gold) e no SQL Server

In [0]:
(
    df_kpi_feriado_dia.write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(GOLD_WRITE_MODE)
    .save(GOLD_FERIADO_DIA_PATH)
)

write_sql_table_typed(df_kpi_feriado_dia, SQL_TABLE_GOLD_FERIADO_DIA)

print(f"[OK] Tabela '{SQL_TABLE_GOLD_FERIADO_DIA}' gravada.")